In [1]:
import os, pickle, numpy as np, pandas as pd
from scipy import sparse
from implicit.als import AlternatingLeastSquares

ALS_DIR   = "C:/OPS/hybrid_filtering/files/ALS"
MODEL_PKL = os.path.join(ALS_DIR, "als_full.pkl")
os.makedirs(ALS_DIR, exist_ok=True)

# ─────────────────────────────────────────────
# A. 데이터 준비
# ─────────────────────────────────────────────
df = pd.read_csv("processed/processed_data_with_actr_disp.csv")
df.dropna(subset=["smry"], inplace=True)

# view_ratio → weight
bins   = [0, 0.25, 0.5, 0.75, 1.0]
labels = [1., 2., 3., 4.]
df["weight"] = pd.cut(df["view_ratio"], bins=bins,
                      labels=labels, include_lowest=True).astype(float)

In [6]:
# ─────────────────────────────────────────────
# B. ALS 학습 & 저장 (없으면 학습, 있으면 건너뜀)
# ─────────────────────────────────────────────
if not os.path.exists(MODEL_PKL):
    print("📌 모델이 없으므로 새로 학습합니다…")

    # 1) 매핑
    user_map = {u: i for i, u in enumerate(df["user_index"].unique())}
    item_map = {a: i for i, a in enumerate(df["asset_id"].unique())}

    df["u_idx"] = df["user_index"].map(user_map)
    df["i_idx"] = df["asset_id"].map(item_map)

    # 2) 희소 행렬
    item_user = sparse.coo_matrix(
        (df["weight"].astype(np.float32),
         (df["i_idx"], df["u_idx"])),
        shape=(len(item_map), len(user_map)),
        dtype=np.float32
    ).tocsr()
    
    user_items = item_user.T

    # 3) 모델 학습
    model = AlternatingLeastSquares(
        factors=100, regularization=0.1,
        iterations=50, calculate_training_loss=True, use_gpu=False
    )
    model.fit(item_user, show_progress=True)

    # 4) 모델·매핑 저장
    payload = {
        "model": model,
        "user_map": user_map,
        "item_map": item_map,
        "bucket_bins": bins,
    }
    with open(MODEL_PKL, "wb") as f:
        pickle.dump(payload, f)
    print(f"✅ 모델 저장 완료 → {MODEL_PKL}")

else:
    print("✅ 기존 모델이 이미 존재하여 학습 단계 생략")

# ─────────────────────────────────────────────
# C. 모델·매핑 불러오기
# ─────────────────────────────────────────────
with open(MODEL_PKL, "rb") as f:
    payload = pickle.load(f)

als_model = payload["model"]
user_map  = payload["user_map"]
item_map  = payload["item_map"]
item_map_inv = {i: a for a, i in item_map.items()}

# (필요 시) 데이터프레임을 매핑과 동기화
df = df[df["user_index"].isin(user_map) & df["asset_id"].isin(item_map)].copy()
df["u_idx"] = df["user_index"].map(user_map)
df["i_idx"] = df["asset_id"].map(item_map)

# item_user / user_items 재생성
item_user = sparse.coo_matrix(
    (df["weight"].astype(np.float32),
     (df["i_idx"], df["u_idx"])),
    shape=(len(item_map), len(user_map)),
    dtype=np.float32
).tocsr()
user_items = item_user.T.tocsr()     # (users × items) 형태

✅ 기존 모델이 이미 존재하여 학습 단계 생략


In [7]:
# ================================================================
# E. 콘텐츠 데이터 전처리 · 형태소 분석
# ================================================================
import os, time, json, pickle
import pandas as pd
from mecab import MeCab

TFIDF_DIR = "C:/OPS/hybrid_filtering/files/TF-IDF"
os.makedirs(TFIDF_DIR, exist_ok=True)

# 1) 콘텐츠 원본 추출 & 중복 제거
content_df = (
    df[["asset_id", "super_asset_nm", "smry", "genre", "actr_disp"]]
    .drop_duplicates()
)

# 2) 그룹화(시리즈·시즌 → 하나의 group_id)
def group_content(df_):
    grp = (
        df_.groupby(["super_asset_nm", "smry"])["asset_id"]
        .apply(list).reset_index()
    )
    grp["group_id"] = range(len(grp))
    id2grp = {
        aid: gid
        for _, (ids, gid) in grp[["asset_id", "group_id"]].iterrows()
        for aid in ids
    }
    df_ = df_.copy()
    df_["group_id"] = df_["asset_id"].map(id2grp)
    return df_, grp

content_df, grp_meta = group_content(content_df)
rep_df   = content_df.drop_duplicates("group_id")
rep_map  = rep_df.set_index("group_id").to_dict("index")

# 3) 형태소 분석 (MeCab)
mecab = MeCab()
stopwords = {"이","그","저","등","를","을","에","에서","하다","있다","되다", "그리고",...}

def analyze_text(text:str) -> str:
    if pd.isna(text) or text == "":
        return ""
    words = [
        w for w, t in mecab.pos(text)
        if t.startswith(("NN","VA","VV")) and len(w) > 1 and w not in stopwords
    ]
    return " ".join(words)

def build_document(row):
    genre_twice = (row["genre"] + " " + row["genre"]) if pd.notna(row["genre"]) else ""
    return " ".join([
        analyze_text(row["smry"] or ""),
        analyze_text(genre_twice),
        analyze_text(row["actr_disp"] or "")
    ])

docs_path = os.path.join(TFIDF_DIR, "analyzed_texts.pkl")
if not os.path.exists(docs_path):
    analyzed_texts = {
        gid: build_document(pd.Series(info))
        for gid, info in rep_map.items()
    }
    with open(docs_path, "wb") as f:
        pickle.dump(analyzed_texts, f)
    print(f"✅ 형태소 분석 완료 → {docs_path}")
else:
    with open(docs_path, "rb") as f:
        analyzed_texts = pickle.load(f)
    print(f"🔄 형태소 분석 스킵 (이미 {len(analyzed_texts)}개 저장)")

# ================================================================
# F. TF-IDF 벡터화 및 저장
# ================================================================
vec_path = os.path.join(TFIDF_DIR, "tfidf_vectorizer.pkl")
gv_path  = os.path.join(TFIDF_DIR, "group_vectors.pkl")

if not os.path.exists(vec_path):
    from sklearn.feature_extraction.text import TfidfVectorizer
    group_ids  = list(analyzed_texts.keys())
    documents  = [analyzed_texts[gid] for gid in group_ids]

    tfidf = TfidfVectorizer(
        min_df=3, max_df=0.9,
        max_features=5000, ngram_range=(1,2)
    )
    tfidf_matrix = tfidf.fit_transform(documents)

    with open(vec_path, "wb") as f:
        pickle.dump(tfidf, f)
    with open(gv_path, "wb") as f:
        pickle.dump({gid: tfidf_matrix[i] for i, gid in enumerate(group_ids)}, f)
    print(f"✅ TF-IDF 학습·저장 완료 ({tfidf_matrix.shape})")
else:
    with open(vec_path, "rb") as f:
        tfidf = pickle.load(f)
    with open(gv_path, "rb") as f:
        group_vectors = pickle.load(f)
    tfidf_matrix = sparse.vstack([group_vectors[g] for g in group_vectors])
    group_ids = list(group_vectors.keys())
    print(f"🔄 TF-IDF 로드 완료 ({tfidf_matrix.shape})")

# group_vectors 딕트가 메모리에 없다면 다시 만들기
if "group_vectors" not in locals():
    group_vectors = {gid: tfidf_matrix[i] for i, gid in enumerate(group_ids)}

# ================================================================
# G. ALS 추천 함수 (안전 버전)
# ================================================================
def get_als_recommendations(user_index:int, N:int=100):
    """ALS 기반 상위 N개 asset_id 반환 (매핑 불일치 안전 처리)"""
    u_idx = user_map[user_index]
    item_ids, scores = als_model.recommend(
        u_idx, user_items[u_idx],
        N=N, filter_already_liked_items=True
    )
    rec_asset_ids, rec_scores = [], []
    for idx, sc in zip(item_ids, scores):
        aid = item_map_inv.get(idx)
        if aid is not None:
            rec_asset_ids.append(aid)
            rec_scores.append(sc)
    return rec_asset_ids, np.array(rec_scores)


🔄 형태소 분석 스킵 (이미 194379개 저장)
🔄 TF-IDF 로드 완료 ((194379, 5000))


In [8]:

# ================================================================
# H. 하이브리드 추천 함수 (ALS + TF-IDF)
# ================================================================
from sklearn.metrics.pairwise import cosine_similarity

asset_to_group = content_df.set_index("asset_id")["group_id"].to_dict()    # ★추가
df["group_id"] = df["asset_id"].map(asset_to_group)  

# 시청 이력·그룹 이력
user_watched = df.groupby("user_index")["asset_id"].apply(list).to_dict()
user_groups  = df.groupby("user_index")["group_id"].apply(
    lambda x: list(set(x))
).to_dict()
asset_to_group = content_df.set_index("asset_id")["group_id"].to_dict()
asset_to_super = content_df.set_index("asset_id")["super_asset_nm"].to_dict()

def hybrid_recommend(user_index:int, *,
                     N=100, M=100, K=10, alpha=0.5):
    """
    ALS(협업필터링) 점수와 TF-IDF(콘텐츠) 유사도를 혼합.
    - N: ALS 후보 개수
    - M: 콘텐츠 기반 후보 개수
    - K: 최종 반환 개수
    - alpha: ALS·콘텐츠 가중치 (0~1)
    """
    watched         = set(user_watched.get(user_index, []))
    user_group_ids  = user_groups.get(user_index, [])

    # ----- 1) ALS 후보 -----
    als_assets, _   = get_als_recommendations(user_index, N=N)

    # ----- 2) 콘텐츠 후보 -----
    if user_group_ids:
        user_vecs = sparse.vstack([group_vectors[g] for g in user_group_ids])
        sims      = cosine_similarity(tfidf_matrix, user_vecs).max(axis=1)
        rank_idx  = np.argsort(sims)[::-1]
        rank_idx  = [i for i in rank_idx if group_ids[i] not in user_group_ids]
        top_groups = [group_ids[i] for i in rank_idx[:M]]
        content_assets = [rep_map[g]["asset_id"] for g in top_groups]
    else:
        sims            = np.zeros(len(group_ids))
        content_assets  = []

    # ----- 3) 후보 통합 & 시청 필터 -----
    candidates = list(set(als_assets + content_assets) - watched)
    if not candidates:
        return []

    # ----- 4) ALS 스코어 (정규화) -----
    u_idx   = user_map[user_index]
    uf      = als_model.user_factors[u_idx]
    cf_idx  = [item_map[a] for a in candidates]
    cf_mat  = als_model.item_factors[cf_idx]
    als_raw = (uf @ cf_mat.T).flatten()
    als_norm = (
        (als_raw - als_raw.min()) / (als_raw.ptp() if als_raw.ptp() else 1)
    )

    # ----- 5) 콘텐츠 스코어 -----
    cont_scores = np.array([
        (sims[asset_to_group[a]] if a in asset_to_group else 0)
        for a in candidates
    ])

    # ----- 6) 하이브리드 & 상위 K ----- 
    hybrid = alpha * als_norm + (1 - alpha) * cont_scores
    order  = np.argsort(hybrid)[::-1]

    seen_super, result = set(), []
    for idx in order:
        aid = candidates[idx]
        sup = asset_to_super.get(aid)
        if sup and sup not in seen_super:
            result.append((aid, sup, hybrid[idx]))
            seen_super.add(sup)
            if len(result) == K:
                break
    return result

In [ ]:
# ================================================================
# I. 사용 예시
# ================================================================
sample_user = df["user_index"].iloc[3]
print(f"\n▶ user_index = {sample_user}")
recs = hybrid_recommend(sample_user, N=100, M=100, K=10, alpha=0.5)

print("추천 결과:")
for aid, sup, sc in recs:
    print(f"  asset_id = {aid:<8} | 프로그램명 = {sup:<20} | 유사도 = {sc:.4f}")



▶ sample user_index = 3
추천 결과:
  asset_id = M4991593LSGI29306301 | 프로그램명 = 바보엄마                 | 유사도 = 0.7003
  asset_id = M4726016LSVJ51151001 | 프로그램명 = 나츠메 우인장 세상과 연을 맺다    | 유사도 = 0.5814
  asset_id = M4860519LSVK03976501 | 프로그램명 = 소림사전기 장경각            | 유사도 = 0.5276
  asset_id = M4270284LSVE62754401 | 프로그램명 = 진시명월                 | 유사도 = 0.5211
  asset_id = M4795356LSGL49106701 | 프로그램명 = 본대로 말하라              | 유사도 = 0.5198
  asset_id = M5170586LSGL93870601 | 프로그램명 = 뷰티풀 데이 인 더 네이버 후드    | 유사도 = 0.5171
  asset_id = M4884495LSGF53872701 | 프로그램명 = 지니강이 플러스 시즌3         | 유사도 = 0.5164
  asset_id = M0226228LSVL33632901 | 프로그램명 = 블리치6기                | 유사도 = 0.5159
  asset_id = M4920939LSVL28614501 | 프로그램명 = 대진제국 4  대진부          | 유사도 = 0.5105
  asset_id = M4993398LFOI22933501 | 프로그램명 = 탁주TV 시즌7             | 유사도 = 0.5085


In [17]:
df[df['user_index'] == 3]['super_asset_nm'].unique()

array(['꼬리에꼬리를무는그날이야기', '나 혼자산다', '2022 일요특집 루틴왕', '범죄도시3', '건축탐구 집'],
      dtype=object)